In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from src.models.decoder_model import DecoderModel

# Parámetros del modelo
input_dim = 12     # features de las acciones
num_heads = 8
head_dim = 16
num_layers = 8

# Crear modelo
model = DecoderModel(
    input_dim=input_dim,
    num_heads=num_heads,
    head_dim=head_dim,
    num_layers=num_layers,
    dropout_rate=0
)

In [9]:
from src.training import load_model

model = load_model(model, "decoder_cl.pth")

In [13]:
from src.solvers import BSGSolver, VCSSolver, ModelSolver

#eval_bsg = BSGSolver().solve(instance_file="benchmarks/BR4.txt", instance_number=1, w=8)
eval_vcs = VCSSolver().solve(instance_file="benchmarks/BR4.txt", instance_number=1)
eval_model = ModelSolver(model).solve(instance_file="benchmarks/BR4.txt", instance_number=1, w=8)

#print(eval_bsg)
print(eval_vcs)
print(eval_model)

92.513621
86.89646921195363


In [ ]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor, as_completed

# --- Configuración ---
instance_file = "benchmarks/BR4.txt"
num_instances = 10
num_threads = 10  # Ajustá según tus cores o I/O paralelismo

# --- Función auxiliar que ejecuta una iteración ---
def run_instance(i):
    try:
        eval_bsg = BSGSolver().solve(instance_file=instance_file, instance_number=i, w=8)
        eval_vcs = VCSSolver().solve(instance_file=instance_file, instance_number=i)
        eval_model = ModelSolver(model).solve(instance_file=instance_file, instance_number=i, w=64)

        return [i, eval_bsg, eval_vcs, eval_model]
    except Exception as e:
        # Importante: no dejar que un error frene todo
        print(f"Error en instancia {i}: {e}")
        return [i, None, None, None]

# --- Ejecución en paralelo ---
results = []
with ProcessPoolExecutor(max_workers=num_threads) as executor:
    futures = [executor.submit(run_instance, i) for i in range(num_instances)]

    for future in as_completed(futures):
        result = future.result()
        results.append(result)

# --- Construcción del DataFrame ---
df = pd.DataFrame(results, columns=['Iteration', 'BSG Cost', 'VCS Cost', 'Model Cost'])

In [11]:
df.sort_values(by='Model Cost')

,Iteration,BSG Cost,VCS Cost,Model Cost
8,17,95.336152,77.032671,76.739139
66,73,93.728043,91.152617,77.326053
1,7,94.840985,84.900670,79.992090
84,91,96.346830,91.351536,82.087166
81,86,96.234007,89.415729,82.105113
...,...,...,...,...
58,60,96.906249,90.531718,91.936417
7,6,97.162031,92.563017,92.195642
99,55,97.770367,93.704487,92.242835
54,49,96.906043,92.759098,92.248152


In [8]:
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- Configuración ---
instance_file = "benchmarks/BR4.txt"
num_instances = 100
num_threads = 10  # Ajustá según tus cores o I/O paralelismo

# --- Función auxiliar que ejecuta una iteración ---
def run_instance(i):
    try:
        eval_vcs = VCSSolver().solve(instance_file=instance_file, instance_number=i)
        eval_model = ModelSolver(model).solve(instance_file=instance_file, instance_number=i, w=8)

        return [i, eval_vcs, eval_model]
    except Exception as e:
        # Importante: no dejar que un error frene todo
        print(f"Error en instancia {i}: {e}")
        return [i, None, None, None]

# --- Ejecución en paralelo ---
results = []
with ThreadPoolExecutor(max_workers=num_threads) as executor:
    futures = [executor.submit(run_instance, i) for i in range(num_instances)]

    for future in as_completed(futures):
        result = future.result()
        results.append(result)

# --- Construcción del DataFrame ---
df = pd.DataFrame(results, columns=['Iteration', 'VCS Cost', 'Model Cost'])

In [9]:
df.sort_values(by='Model Cost')

,Iteration,VCS Cost,Model Cost
58,43,89.126160,75.253576
8,17,77.032671,77.032596
87,88,90.482027,80.124030
18,20,81.777277,81.777425
99,91,91.351536,85.133069
...,...,...,...
51,55,93.704487,93.902212
5,3,92.929990,94.067053
80,78,94.831892,94.311656
60,64,92.626706,94.462871


In [10]:
df.mean()

Iteration     49.500000
VCS Cost      90.787513
Model Cost    90.585142
dtype: float64